In [0]:
%pip install ../
dbutils.library.restartPython()

In [0]:
%pip install -U openai pandas dspy-ai pydantic
dbutils.library.restartPython()

In [0]:
CATALOG = "rohitb_demo"
SCHEMA = "spirit_demo"
REVIEWS_TABLE = "spirit_google_reviews"
QUESTIONS_TABLE = "questions"
TARGET_TABLE = "spirit_google_reviews_predictions_original"
WORKSPACE_URL = f'https://{spark.conf.get("spark.databricks.workspaceUrl")}'
MODEL_ID = "databricks-meta-llama-3-1-70b-instruct"
BATCH_ETL_BATCH_SIZE = 10 # make this 1000 or 10000 what ever batch size you can afford to fail on
VOLUME_BASE_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/costa_volume" # modify this if you want to change the catalog or schema, volumes are at schema level
print("Variable: CATALOG = {}".format(CATALOG))
print("Variable: SCHEMA = {}".format(SCHEMA))
print("Variable: QUESTIONS_TABLE = {}".format(QUESTIONS_TABLE))
print("Variable: REVIEWS_TABLE = {}".format(REVIEWS_TABLE))
print("Variable: TARGET_TABLE = {}".format(TARGET_TABLE))
print("Variable: WORKSPACE_URL = {}".format(WORKSPACE_URL))
print("Variable: MODEL_ID = {}".format(MODEL_ID))
print("Variable: BATCH_ETL_BATCH_SIZE = {}".format(BATCH_ETL_BATCH_SIZE))
print("Variable: VOLUME_BASE_PATH = {}".format(VOLUME_BASE_PATH))

In [0]:
from auto_topic.domains import DomainConfigTable, Domain

dct = DomainConfigTable(catalog=CATALOG, schema=SCHEMA, table=QUESTIONS_TABLE)


## Define Domains

**Define different domains and questions you want answered. You can have a lot of these and the llm will do a two pass, first identify the topic and then map to fill out all the information that needs to be filled out in the domain**

Each domain will require:
1. topic -> 1 word all lower case
2. when ->  topic condition: succinct sentence aligning topic name with specific entities aligning with this topic, e.g. in retail topic name defect may align with sneaker but in manufacturing topic name 
3. details -> single sentence expression what you want extracted (just additional info)

You can optionally extract additional info in each topic using the `.with_additional_info` with the following two fields:
1. item_name -> which is only alpha numeric json key for labeling the concept/question
2. item_description -> the details in how to extract the item/concept/detail

For example when someone is describing a review and if they describe the item was used as a gift you may want to extract that detail if you think this may be something that you can use to suggest specific items as great gifts during a specific season.


In [0]:
dct.with_topic(
    topic=Domain(
        topic="check_in_bag_drop",
        when="Mentions of check-in, bag drop, or kiosk usability.",
        details="Assessment of check-in and baggage drop."
    ).with_additional_info(
        item_name="self_kiosk",
        item_description="Ease of kiosk use <easy/difficult/unavailable>"
    ).with_additional_info(
        item_name="staff_assistance",
        item_description="Helpfulness of staff <helpful/unhelpful>"
    ).with_additional_info(
        item_name="bag_drop_efficiency",
        item_description="Baggage drop-off process <smooth/delayed/problematic>"
    )
)

dct.with_topic(
    topic=Domain(
        topic="boarding",
        when="Experiences with boarding process, priority boarding, or crowd control.",
        details="Evaluation of boarding efficiency."
    ).with_additional_info(
        item_name="gate_organization",
        item_description="Clarity of announcements <clear/confusing/missing>"
    ).with_additional_info(
        item_name="timeliness",
        item_description="Punctuality of boarding <on-time/delayed/disorganized>"
    ).with_additional_info(
        item_name="staff_attitude",
        item_description="Gate staff attitude <friendly/rude/indifferent>"
    )
)

dct.with_topic(
    topic=Domain(
        topic="in_flight_experience",
        when="Mentions of seat comfort, cleanliness, service.",
        details="Assessment of in-flight experience."
    ).with_additional_info(
        item_name="seat_comfort",
        item_description="Seat quality and legroom <comfortable/tight/unbearable>"
    ).with_additional_info(
        item_name="cleanliness",
        item_description="Cabin cleanliness <clean/dirty/filthy>"
    ).with_additional_info(
        item_name="crew_service",
        item_description="Flight attendant service <excellent/poor>"
    )
)

dct.with_topic(
    topic=Domain(
        topic="gate_experience",
        when="Feedback on gate service, announcements, assistance.",
        details="Evaluation of gate experience."
    ).with_additional_info(
        item_name="gate_agent_attitude",
        item_description="Gate staff attitude <helpful/rude/unavailable>"
    ).with_additional_info(
        item_name="boarding_announcement",
        item_description="Clarity of instructions <clear/confusing/missing>"
    ).with_additional_info(
        item_name="delay_communication",
        item_description="Effectiveness of delay updates <transparent/misleading/no_info>"
    )
)

dct.with_topic(
    topic=Domain(
        topic="flight_delays",
        when="Mentions of delays, cancellations, and airline handling.",
        details="Assessment of delay handling."
    ).with_additional_info(
        item_name="delay_reason_given",
        item_description="Communication of delay reasons <clear/vague/nonexistent>"
    ).with_additional_info(
        item_name="compensation_offered",
        item_description="Resolution for delays/cancellations <fair/none/unsatisfactory>"
    ).with_additional_info(
        item_name="alternative_flight",
        item_description="Rebooking experience <smooth/problematic>"
    )
)

dct.with_topic(
    topic=Domain(
        topic="baggage_handling",
        when="Complaints about lost, damaged baggage, or claim process.",
        details="Evaluation of baggage management."
    ).with_additional_info(
        item_name="baggage_lost",
        item_description="Lost baggage <yes/no>"
    ).with_additional_info(
        item_name="damage_reported",
        item_description="Baggage condition <intact/damaged/severely_damaged>"
    ).with_additional_info(
        item_name="claim_processing",
        item_description="Ease of claim filing <easy/difficult>"
    )
)

dct.with_topic(
    topic=Domain(
        topic="customer_service",
        when="Feedback on policies, support, and issue resolution.",
        details="Evaluation of customer service."
    ).with_additional_info(
        item_name="staff_responsiveness",
        item_description="Staff response to issues <helpful/unhelpful>"
    ).with_additional_info(
        item_name="policy_flexibility",
        item_description="Strictness of policies <reasonable/strict/unfair>"
    ).with_additional_info(
        item_name="refund_experience",
        item_description="Ease of refunds <smooth/difficult/impossible>"
    )
)

dct.with_topic(
    topic=Domain(
        topic="value_for_money",
        when="Comments on pricing, add-on fees, and perceived value.",
        details="Passenger perception of value."
    ).with_additional_info(
        item_name="ticket_price",
        item_description="Ticket pricing perception <fair/overpriced/budget-friendly>"
    ).with_additional_info(
        item_name="hidden_fees",
        item_description="Experience with extra charges <none/minor/excessive>"
    ).with_additional_info(
        item_name="overall_satisfaction",
        item_description="General value assessment <good/poor/neutral>"
    )
)

In [0]:
dct.setup(spark)
spark.table(f"{dct.catalog}.{dct.schema}.{dct.table}").display()

# Batch ETL

In [0]:
from auto_topic.domains import DomainConfigTable
from auto_topic.sentiment import get_analyzer, enable_arize_tracing, get_valid_responses_for_categories, get_when_to_use_category, build_analysis_views
import pandas as pd

dct = DomainConfigTable.from_table(spark, catalog=CATALOG, schema=SCHEMA, table=QUESTIONS_TABLE)

TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().getOrElse(None)
topics_df = pd.DataFrame([topic.to_kwargs() for topic in dct.topics])

topics_df

In [0]:
# MODEL_ID = 'databricks-meta-llama-3-1-70b-instruct'
# # MODEL_ID = "databricks-meta-llama-3-1-405b-instruct"
# # MODEL_ID = 'rb-llama80b-temp'

In [0]:
from pyspark.sql import functions as F
import json
import dspy

from typing import Iterator, Tuple

@F.pandas_udf("string")
def extract_domain_details(feedback_and_ratings: Iterator[Tuple[pd.Series, pd.Series]]) -> Iterator[pd.Series]:
    # Do some expensive initialization with a state
    language_model = dspy.LM(
        model=f"databricks/{MODEL_ID}",
        max_tokens=1000,
        temperature=0.1,
        api_key=TOKEN,
        api_base=f"{WORKSPACE_URL}/serving-endpoints/"
    )
    extract = get_analyzer(topics_df, language_model)
    for feedback_arr, rating_arr in feedback_and_ratings:
        # Use that state for whole iterator.
        feedbacks, ratings = feedback_arr.tolist(), rating_arr.tolist()
        results = []
        for feedback, rating in zip(feedbacks, ratings):
            resp = extract(feedback=feedback, rating=str(rating))
            final_resp = json.dumps({"category_breakdown": resp.breakdown.to_dict(), 
                    "category_selection": resp.category_selection,
                    "category_selection_rationale": resp.category_selection_rationale,
                    "all_categories": get_valid_responses_for_categories(topics_df)})
            results.append(final_resp)

        yield pd.Series(results)
  


In [0]:
reviews = spark.table(f"{CATALOG}.{SCHEMA}.{REVIEWS_TABLE}").limit(2)
reviews = reviews.withColumn("analysis", extract_domain_details("review", "rating"))
reviews.display()

In [0]:
#  REMOVE LIMIT 10
if spark.catalog.tableExists(f"{CATALOG}.{SCHEMA}.{TARGET_TABLE}") is False:
    print(f"Creating Target Table: {CATALOG}.{SCHEMA}.{TARGET_TABLE}")
    spark.sql(f"""
              SELECT *, cast(null as string) as analysis FROM {CATALOG}.{SCHEMA}.{REVIEWS_TABLE}
               
              """).write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.{TARGET_TABLE}")
else:
    print(f"Table: {CATALOG}.{SCHEMA}.{TARGET_TABLE} already exists!")

# seed reviews
display(spark.sql(f"SELECT * FROM {CATALOG}.{SCHEMA}.{TARGET_TABLE};"))


# ETL Logic

1. Identify unanalyzed columns (where analysis is null)
2. Then chunk the amount of rows to process via BATCH_ETL_BATCH_SIZE and then commit the transaction via a merge.
3. Repeat till the analysis columns are not null


If there are 1000 null records and BATCH_ETL_BATCH_SIZE=500 you will have 2 transcations made to the table. 

In [0]:
from delta.tables import DeltaTable

def get_unanalyzed_records(spark, batch_size = None):
    if batch_size is None:
        return spark.table(f"{CATALOG}.{SCHEMA}.{TARGET_TABLE}").where("analysis is null")
    else:
        return spark.table(f"{CATALOG}.{SCHEMA}.{TARGET_TABLE}").where("analysis is null").limit(batch_size)

unanalyzed_records = get_unanalyzed_records(spark, BATCH_ETL_BATCH_SIZE)
unanalyzed_records_ct = unanalyzed_records.count()
unanalyzed_records_ct
batch_ct = 1
while unanalyzed_records_ct > 0:
    print(f"Analyzing {unanalyzed_records_ct} records...; batch number {batch_ct}")
    analyzed_records = unanalyzed_records.withColumn("analysis", extract_domain_details("review", "rating"))
    target_table = DeltaTable.forName(spark, f"{CATALOG}.{SCHEMA}.{TARGET_TABLE}")
    target_table.alias("target").merge(
        source=analyzed_records.alias("source"),
        condition="target.review_id = source.review_id",
    ).whenMatchedUpdateAll().execute()

    # fetch new batch
    unanalyzed_records = get_unanalyzed_records(spark, BATCH_ETL_BATCH_SIZE)
    unanalyzed_records_ct = unanalyzed_records.count()
    batch_ct += 1

In [0]:
display(unanalyzed_records)

%md
## Generate Analysis Views

Build views to analyze data along with comments. These tables can be used for genie data room

In [0]:
build_analysis_views(
    spark=spark,
    catalog=CATALOG,
    schema=SCHEMA,
    analysis_table=TARGET_TABLE, 
    primary_key_col_name="review_id", 
    domain_config_table=dct
)

In [0]:
spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}").display()


## Identify errors

Identify failed analysis tables by looking for `WHERE analysis:category_breakdown:error is not null`

In [0]:
# Failures in the analysis column
display(spark.sql(f"""
SELECT * FROM {CATALOG}.{SCHEMA}.{TARGET_TABLE}
WHERE analysis:category_breakdown:error is not null;
"""))